In [1]:
from bcbench.config import get_config
from bcbench.dataset import DatasetEntry, load_dataset_entries

_config = get_config()

bcbench_dataset: list[DatasetEntry] = load_dataset_entries(_config.paths.dataset_path)

print(f"Total number of instances: {len(bcbench_dataset)}")

Total number of instances: 84


In [2]:
from collections import Counter

import pandas as pd

project_counts: Counter[str] = Counter([entry.extract_project_name().replace(" ", "") for entry in bcbench_dataset])
project_df = pd.DataFrame(project_counts.items(), columns=["Project", "Count"]).sort_values("Count", ascending=False)
project_df["Percentage"] = (project_df["Count"] / project_df["Count"].sum() * 100).round(1)

print("Projects Distribution:")
print(project_df.to_string(index=False))

Projects Distribution:
                   Project  Count  Percentage
                   BaseApp     68        81.0
                   Shopify      7         8.3
            Sustainability      2         2.4
       SubscriptionBilling      2         2.4
              ExcelReports      2         2.4
EssentialBusinessHeadlines      1         1.2
   EnforcedDigitalVouchers      1         1.2
     AutomaticAccountCodes      1         1.2


In [3]:
area_counts: Counter[str] = Counter([entry.metadata.area if entry.metadata.area else "Unknown" for entry in bcbench_dataset])

area_df = pd.DataFrame(area_counts.items(), columns=["Area", "Count"]).sort_values("Count", ascending=False)
area_df["Percentage"] = (area_df["Count"] / area_df["Count"].sum() * 100).round(1)

print("\nArea Distribution:")
print(area_df.to_string(index=False))


Area Distribution:
                Area  Count  Percentage
           inventory     16        19.0
             finance     14        16.7
               sales     11        13.1
             shopify      7         8.3
             project      5         6.0
           warehouse      5         6.0
       manufacturing      4         4.8
             service      3         3.6
      sustainability      2         2.4
           reporting      2         2.4
             pricing      2         2.4
subscription billing      2         2.4
            workflow      2         2.4
            assembly      2         2.4
                 crm      2         2.4
       visualization      1         1.2
        intercompany      1         1.2
            eservice      1         1.2
           utilities      1         1.2
           purchases      1         1.2


In [4]:
from unidiff import PatchSet

gold_patch_sets: list[PatchSet] = [PatchSet(entry.patch) for entry in bcbench_dataset]

files_changed_counts: Counter[int] = Counter([len(patch_set) for patch_set in gold_patch_sets])

files_changed_df = pd.DataFrame(files_changed_counts.items(), columns=["Number of Files Changed", "Count"]).sort_values("Number of Files Changed")
files_changed_df["Percentage"] = (files_changed_df["Count"] / files_changed_df["Count"].sum() * 100).round(1)

print(files_changed_df.to_string(index=False))

 Number of Files Changed  Count  Percentage
                       1     69        82.1
                       2     12        14.3
                       3      1         1.2
                       4      1         1.2
                       6      1         1.2


In [5]:
lines_of_code_changed: list[int] = [patch_set.added + patch_set.removed for patch_set in gold_patch_sets]

bins = [0, 5, 10, 20, 50, 100, float("inf")]
labels = ["1-5", "6-10", "11-20", "21-50", "51-100", "100+"]
bucketed = pd.cut(lines_of_code_changed, bins=bins, labels=labels)

lines_of_code_changed_df = pd.DataFrame({"LoC": bucketed}).value_counts().reset_index(name="Count")
lines_of_code_changed_df = lines_of_code_changed_df.sort_values("LoC")
lines_of_code_changed_df["Percentage"] = (lines_of_code_changed_df["Count"] / lines_of_code_changed_df["Count"].sum() * 100).round(1)

print(lines_of_code_changed_df.to_string(index=False))

   LoC  Count  Percentage
   1-5     36        42.9
  6-10      8         9.5
 11-20     14        16.7
 21-50     17        20.2
51-100      5         6.0
  100+      4         4.8


In [6]:
from statistics import mean

files_changed: list[int] = [len(patch_set) for patch_set in gold_patch_sets]

print("Solution requires:")
print(f"~{mean(lines_of_code_changed):.1f} LoC on average (stdev: {pd.Series(lines_of_code_changed).std():.1f}, median: {pd.Series(lines_of_code_changed).median():.1f})")
print(f"~{mean(files_changed):.1f} files on average (stdev: {pd.Series(files_changed).std():.1f}, median: {pd.Series(files_changed).median():.1f})")

Solution requires:
~19.6 LoC on average (stdev: 26.8, median: 10.0)
~1.3 files on average (stdev: 0.7, median: 1.0)


In [7]:
image_counts: list[int] = [entry.metadata.image_count or 0 for entry in bcbench_dataset]

bins = [-1, 0, 5, 10, float("inf")]
labels = ["0", "1-5", "6-10", "10+"]
bucketed = pd.cut(image_counts, bins=bins, labels=labels)

image_count_df = pd.DataFrame({"Image Count": bucketed}).value_counts().reset_index(name="Count")
image_count_df = image_count_df.sort_values("Image Count")
image_count_df["Percentage"] = (image_count_df["Count"] / image_count_df["Count"].sum() * 100).round(1)

print(image_count_df.to_string(index=False))

Image Count  Count  Percentage
          0     29        34.5
        1-5     24        28.6
       6-10     22        26.2
        10+      9        10.7
